In [1]:
# ============================================================
# Local: RL for simplifying "hard" unknots (CSV first-col only)
# ============================================================

# --------- 0) Install dependencies ----------
# !pip -q install "gymnasium>=0.29" "stable-baselines3>=2.3.0" spherogram gcsfs numpy tqdm
# !pip -q install snappy-manifolds

# --------- 1) Imports & seeds ----------
import os, re, json, random, io, gzip, csv, itertools
import numpy as np
from typing import List, Iterable, Optional
from dataclasses import dataclass

import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback

from spherogram import Link
import gcsfs

from pathlib import Path
from stable_baselines3.common.logger import configure

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [2]:
# --------- 2) CSV → GCS reader (first column only) ----------
def _normalize_gcs_path(path: str) -> str:
    # gcsfs usually expects "bucket/path". Accept "gs://bucket/path" as well.
    return path[5:] if path.startswith("gs://") else path

def read_first_column_gcs(path: str,
                          max_lines: int | None = None,
                          has_header: bool = True,
                          delimiter: str | None = None,
                          encoding: str = "utf-8") -> list[str]:
    """
    Read a CSV (optionally .gz) from GCS and return only the first column as strings.
    - path: e.g. "gs://bucket/folder/file.csv" or "bucket/folder/file.csv"
    - max_lines: cap rows returned (after header handling)
    - has_header: skip first row if True
    - delimiter: set explicitly ("," or "\t"); if None, sniff a small sample
    - encoding: for non-gz files
    """
    fs = gcsfs.GCSFileSystem(token="anon")
    path = _normalize_gcs_path(path)
    results: list[str] = []

    def make_reader(fh):
        nonlocal delimiter
        if delimiter is None:
            pos = fh.tell()
            sample = fh.read(4096)
            fh.seek(pos)
            try:
                dialect = csv.Sniffer().sniff(sample)
            except Exception:
                dialect = csv.excel  # default (,)
            return csv.reader(fh, dialect)
        else:
            return csv.reader(fh, delimiter=delimiter)

    if path.lower().endswith(".gz"):
        with fs.open(path, "rb") as fb:
            with gzip.GzipFile(fileobj=fb, mode="rb") as gz:
                with io.TextIOWrapper(gz, encoding=encoding, newline="") as ft:
                    reader = make_reader(ft)
                    if has_header:
                        next(reader, None)
                    for i, row in enumerate(reader):
                        if max_lines is not None and i >= max_lines: break
                        if row:
                            results.append(str(row[0]).strip())
    else:
        with fs.open(path, "rt", encoding=encoding, newline="") as f:
            reader = make_reader(f)
            if has_header:
                next(reader, None)
            for i, row in enumerate(reader):
                if max_lines is not None and i >= max_lines: break
                if row:
                    results.append(str(row[0]).strip())
    return results

# --------- 3) Strict PD/DT parser + cleaner ----------
_RE_DT_PREFIX = re.compile(r'^\s*DT\s*:\s*\[', re.I)
_RE_PDLIST = re.compile(r'^\s*\[\s*(\[\s*\d+(?:\s*,\s*\d+){3}\s*\]\s*,?\s*)+\]\s*$') # [[...], ...]
_RE_XPD       = re.compile(r'[Xx]\s*\[')

def parse_link_strict(s: str) -> Link:
    """
    Accept ONLY PD/DT encodings:
      - 'DT: [ ... ]'
      - PD as nested lists: [[8,3,1,4],[2,6,3,5],...]
      - PD in X[...] blocks: X[a,b,c,d], X[...], ...
    Also unwrap simple JSON/JSONL if it contains 'pd' or 'dt' fields.
    """
    t = s.strip()
    if _RE_DT_PREFIX.match(t):
        return Link(t)
    if _RE_PDLIST.match(t):
    # Convert safely into Python object
        try:
            pd_obj = json.loads(t)  # works if it's valid JSON-like
        except json.JSONDecodeError:
            # fallback for python-style lists with no quotes
            import ast
            pd_obj = ast.literal_eval(t)
        return Link(pd_obj)
    if _RE_XPD.search(t):
        try:
            return Link(t)
        except Exception:
            items = re.findall(r'[Xx]\s*\[([^\]]+)\]', t)
            if not items:
                raise
            blocks = []
            for it in items:
                nums = [int(x.strip()) for x in it.split(',')]
                if len(nums) != 4:
                    raise ValueError("PD block must have 4 integers")
                blocks.append(nums)
            return Link(str(blocks))
    if (t.startswith("{") or t.startswith("[")) and not _RE_PDLIST.match(t):
        try:
            obj = json.loads(t)
            for key in ("pd","PD","pd_code","PD_code","dt","DT"):
                if key in obj:
                    return parse_link_strict(obj[key])
        except Exception:
            pass
    raise ValueError("Not a PD/DT code")

def clean_pd_lines(lines: Iterable[str], max_keep: int | None = None) -> List[str]:
    good = []
    for s in lines:
        try:
            _ = parse_link_strict(s)
            good.append(s.strip())
            if max_keep and len(good) >= max_keep:
                break
        except Exception:
            continue
    return good

# --------- 4) Spherogram helpers ----------
def crossings(link: Link) -> int:
    return len(link.crossings)

def is_trivial_zero(link: Link) -> bool:
    return crossings(link) == 0

def can_reduce_basic(L: Link) -> bool:
    before = crossings(L)
    tmp = Link(L.PD_code())
    changed = tmp.simplify(mode="basic")
    return bool(changed and crossings(tmp) < before)

# --- UPDATED: pure RIII only (no RI/RII), in-place on the given Link ---
def riii_shuffle_only_link(link: Link, k: int, tries_per_move: int = 20):
    """
    Apply up to k random Type-III moves using Spherogram internals
    without triggering any RI/RII simplification. Returns (link, done).
    """
    # lazy import so we don't change your global imports
    from spherogram.links import simplify as _simp
    list_fn  = getattr(_simp, "possible_type_III_moves", None)
    apply_fn = getattr(_simp, "reidemeister_III", None)
    if list_fn is None or apply_fn is None:
        return link, 0

    L = link  # operate in-place
    done = 0
    for _ in range(k):
        moves = list_fn(L)
        if not moves:
            break
        tries = min(tries_per_move, len(moves))
        c0 = crossings(L)
        success = False
        for tri in random.sample(moves, tries):
            apply_fn(L, tri)  # in-place RIII
            if crossings(L) == c0:  # ensure it's a pure RIII (no crossing change)
                success = True
                break
        if not success:
            break
        done += 1
    return L, done

# --------- 5) RL Environment (macro actions with blocking + simple ranking reward) ----------
# The action is MultiDiscrete: [mode, cap]
#   mode ∈ {0:basic, 1:level, 2:pickup, 3:backtrack}
#   cap  ∈ {0..cap_max}

maxstepsdone = 500

# 1) Config
@dataclass
class EnvCfg:
    max_steps: int = maxstepsdone
    step_penalty: float = 0.05           # small time cost
    reward_finish: float = 10.0
    allow_backtrack: bool = True
    cap_max: int = 8
    # Simple mode ranking reward: 0 > 1 > 2 > 3
    mode_rewards: tuple[float, float, float, float] = (3.0, 2.0, 1.0, 0.0)
    # Reward shaping weights (dense progress) # TODO: tune these!
    # w_delta: float = 1.0          # reward per crossing removed
    # w_uphill: float = 0.5         # extra penalty per crossing added (beyond losing delta)
    # w_potential: float = 0.02     # small push toward low crossings

class SphKnotEnv(gym.Env):
    # ... keep everything else as you have it ...

    def __init__(self, pd_lines: list[str], cfg: EnvCfg):
        super().__init__()
        self.cfg = cfg
        self.pd_lines = pd_lines
        self.rng = random.Random(SEED)

        self.num_actions = 4 if self.cfg.allow_backtrack else 3
        self.action_space = spaces.MultiDiscrete(
            np.array([self.num_actions, self.cfg.cap_max + 1], dtype=np.int64)
        )
        self.obs_dim = 6
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(self.obs_dim,), dtype=np.float32
        )

        self.L: Optional[Link] = None
        self._steps = 0
        self._last_drop = 0
        self._after_backtrack = False

        # NEW: blocking state (True = blocked). Index by mode.
        self._blocked = [False, False, False, False]

    def _reset_blocks(self):
        self._blocked = [False, False, False, False]

    def _map_blocked_mode(self, mode: int) -> int:
        """If requested mode is blocked, advance to the next unblocked mode (cyclic)."""
        m = mode % self.num_actions
        for _ in range(self.num_actions):
            if not self._blocked[m]:
                return m
            m = (m + 1) % self.num_actions
        # Fallback (shouldn't happen): use backtrack
        return min(3, self.num_actions - 1)

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self._steps = 0
        self._last_drop = 0
        self._after_backtrack = False
        self._reset_blocks()
        for _ in range(10):
            s = self.rng.choice(self.pd_lines)
            try:
                self.L = parse_link_strict(s)
                break
            except Exception:
                self.L = None
        if self.L is None:
            self.L = parse_link_strict(self.pd_lines[0])
        return self._obs(), {"crossings": crossings(self.L)}

    def step(self, action):
        # Unpack action
        self._steps += 1
        if isinstance(action, (list, tuple, np.ndarray)):
            mode_req, cap = int(action[0]), int(action[1])
        else:
            mode_req, cap = int(action), 0
        cap = max(0, min(cap, self.cfg.cap_max))

        # Apply blocking: remap to first unblocked mode (cyclic from requested)
        mode = self._map_blocked_mode(mode_req)

        # Context
        c_before = crossings(self.L)

        # Execute chosen (possibly remapped) mode
        if mode == 0:
            self.L.simplify(mode="basic")

        elif mode == 1:
            steps = (cap if cap > 0 else 1)
            self.L.simplify(mode="level", type_III_limit=steps)

        elif mode == 2:
            steps = (cap if cap > 0 else 1)
            self.L.simplify(mode="pickup", type_III_limit=steps)

        elif mode == 3 and self.num_actions == 4:
            steps = (cap if cap > 0 else 1)
            self.L.backtrack(steps=steps, prob_type_1=0.35, prob_type_2=0.65)
            # optional: tiny RIII shuffle is ok to keep or remove; keeping it is fine
            self.L, _ = riii_shuffle_only_link(self.L, min(steps, 2))

        # Post
        c_after = crossings(self.L)
        delta = c_before - c_after  # success iff delta > 0
        self._last_drop = max(delta, 0)

        # --- Reward: simple ranking by mode, small step penalty
        reward = self.cfg.mode_rewards[mode] - self.cfg.step_penalty
        # --- Reward: dense progress toward fewer crossings # TODO: add back
        # delta = c_before - c_after  (positive is good)
        # reward = (
        #     self.cfg.w_delta * delta
        #     - self.cfg.w_uphill * max(0, -delta)
        #     - self.cfg.w_potential * c_after
        #     - self.cfg.step_penalty
        # )

        # Finish bonus (unchanged)
        done = False
        if is_trivial_zero(self.L):
            reward += self.cfg.reward_finish
            done = True
        if self._steps >= self.cfg.max_steps:
            done = True

        # --- Blocking logic
        if delta > 0:
            # success -> reset blocks
            self._reset_blocks()
        # else: # TODO: consider allowing 0-delta attempts to not block (currently any non-success blocks)
        #   if mode == 3:
        #       self._reset_blocks()
        #   else:
        #       # only block if it actually got worse; allow 0-delta tries
        #       if delta < 0:
        #           self._blocked[mode] = True
        else:
            if mode == 3:
                # backtrack used -> reset to start of cycle
                self._reset_blocks()
            else:
                # unsuccessful non-backtrack -> block this mode
                self._blocked[mode] = True

        # After-backtrack flag (kept for your obs if needed)
        self._after_backtrack = (mode == 3 and self.num_actions == 4)

        info = {
            "crossings": c_after,
            "delta": delta,
            "mode_requested": mode_req,
            "mode_effective": mode,
            "cap": cap,
            "blocked": tuple(self._blocked),
        }
        return self._obs(), reward, done, False, info

In [3]:
# Colab-specific mount disabled for local runs:
# from google.colab import drive
# drive.mount('/content/drive', force_remount=True)


# === Paths (local) ===
BASE = str((Path.cwd() / "crossing-reduction").resolve())
PD_PATH = f"{BASE}/3-16.txt"
SMALL_JONES = f"{BASE}/small-jones.txt"
BEST_MODEL_PATH = f"{BASE}/best_model.zip"   # flat, no subfolder
TB_DIR = f"{BASE}/tb_knots"                  # tensorboard logs here
LOCAL_RANDOM = f"{BASE}/random_diagrams.csv"
LOCAL_RANDOM2 = f"{BASE}/random_diagrams_2.csv"
LOCAL3 = f"{BASE}/very_hard_unknots_2.csv"
os.makedirs(TB_DIR, exist_ok=True)
print(f"Using local BASE: {BASE}")


Using local BASE: /Users/annedranowski/HQ/🔬 Research/Untangling Number/crossing-reduction


In [4]:
# --------- 6) Load mixed CSVs from GCS/local (first column only), then shuffle ----------
# Files from GCS:
GCS_CSV_PATH_MAIN = "gs://gdm-unknotting/hard_unknots.csv"         # large file
GCS_CSV_PATH_VERY = "gs://gdm-unknotting/very_hard_unknots.csv"

# Files from local BASE:
LOCAL_RANDOM = f"{BASE}/random_diagrams.csv"
LOCAL_RANDOM2 = f"{BASE}/random_diagrams_2.csv"
LOCAL3 = f"{BASE}/very_hard_unknots_2.csv"

# Parsing options (set per file if needed)
HAS_HEADER_MAIN = True
HAS_HEADER_VERY = True
HAS_HEADER_RANDOM = True
DELIMITER_MAIN  = None   # None = auto-sniff; else set "," or "\t"
DELIMITER_VERY  = None
MAX_ROWS_MAIN   = 385   # cap from the big file
SEED            = 42

# 1) Read first column from GCS files
raw_main = read_first_column_gcs(
    GCS_CSV_PATH_MAIN,
    max_lines=MAX_ROWS_MAIN,
    has_header=HAS_HEADER_MAIN,
    delimiter=DELIMITER_MAIN,
    encoding="utf-8",
)
raw_very = read_first_column_gcs(
    GCS_CSV_PATH_VERY,
    max_lines=None,
    has_header=HAS_HEADER_VERY,
    delimiter=DELIMITER_VERY,
    encoding="utf-8",
)

def read_first_col_local(path: str, has_header: bool = True, encoding: str = "utf-8") -> list[str]:
    out = []
    with open(path, "r", encoding=encoding, newline="") as f:
        rdr = csv.reader(f)
        if has_header:
            next(rdr, None)
        for row in rdr:
            if row:
                out.append(row[0].strip())
    return out

required_local_csvs = [LOCAL_RANDOM, LOCAL_RANDOM2, LOCAL3]
missing_local_csvs = [f for f in required_local_csvs if not os.path.exists(f)]
assert not missing_local_csvs, (
    "Missing required local CSV file(s):\n - " + "\n - ".join(missing_local_csvs) +
    "\nAdd them under BASE or update LOCAL_* paths."
)

raw_random = read_first_col_local(LOCAL_RANDOM, has_header=HAS_HEADER_RANDOM, encoding="utf-8")
raw_random2 = read_first_col_local(LOCAL_RANDOM2, has_header=HAS_HEADER_RANDOM, encoding="utf-8")
raw3 = read_first_col_local(LOCAL3, has_header=HAS_HEADER_RANDOM, encoding="utf-8")

print(f"[MAIN @ GCS] {len(raw_main)} rows (capped {MAX_ROWS_MAIN}).")
print(f"[VERY @ GCS] {len(raw_very)} rows.")
print(f"[RANDOM @ local] {len(raw_random)} rows.")
print(f"[RANDOM2 @ local] {len(raw_random2)} rows.")
print(f"[RAW3 @ local] {len(raw3)} rows.")

# 2) Concatenate, then clean to PD/DT
combined_raw = list(raw_very) + list(raw_main) + list(raw_random) + list(raw_random2) + list(raw3)
pd_lines = clean_pd_lines(combined_raw, max_keep=None)
assert pd_lines, "No valid PD/DT lines after cleaning—check CSV columns or delimiters."

# 3) Shuffle for training (reproducible)
rng = random.Random(SEED)
rng.shuffle(pd_lines)

print(f"[Data] Kept {len(pd_lines)} PD/DT strings after strict filtering and shuffling.")
# (Optionally, deduplicate; uncomment if needed)
# pd_lines = list(dict.fromkeys(pd_lines))
# print(f"[Data] After de-duplication: {len(pd_lines)} PD/DT strings.")

#pd_lines_train = pd_lines[:TRAIN_ROWS]
#pd_lines_test = pd_lines[TRAIN_ROWS+1:MAX_ROWS]

def _obs_patch(self):
    c = crossings(self.L)
    try:
        comps = len(self.L.link_components)
    except Exception:
        comps = 1
    tmp = Link(self.L.PD_code())
    reduced = tmp.simplify(mode="basic")
    can_reduce = 1.0 if (reduced and crossings(tmp) < c) else 0.0
    recent = 1.0 if getattr(self, "_last_drop", 0) > 0 else 0.0
    return np.array([c, comps, self._steps, can_reduce, recent, 1.0], dtype=np.float32)

# Monkey-patch if needed
SphKnotEnv._obs = getattr(SphKnotEnv, "_obs", _obs_patch)

# --------- 7) Build envs ----------
cfg = EnvCfg(max_steps=maxstepsdone, allow_backtrack=True)
def make_env():
    return SphKnotEnv(pd_lines, cfg)

env = make_env()
vec_env = DummyVecEnv([lambda: make_env()])

obs, info = env.reset()
print("[Env] obs:", obs, "start crossings:", info["crossings"])


[MAIN @ GCS] 385 rows (capped 385).
[VERY @ GCS] 385 rows.
[RANDOM @ local] 770 rows.
[RANDOM2 @ local] 770 rows.
[RAW3 @ local] 385 rows.
[Data] Kept 2695 PD/DT strings after strict filtering and shuffling.
[Env] obs: [35.  1.  0.  0.  0.  1.] start crossings: 35


In [5]:

# The BASE directory is defined in hnGBP8uScIAl
print(f"Listing contents of: {BASE}/")

# List all files and directories in the BASE path
if os.path.exists(BASE):
    for root, dirs, files in os.walk(BASE):
        # Only show immediate children of BASE for brevity unless user specifies more depth
        if root == BASE:
            print("  Directories:")
            for d in dirs:
                print(f"    - {d}/")
            print("  Files:")
            for f in files:
                print(f"    - {f}")
            break # Only list the top level of BASE for now
else:
    print(f"The directory '{BASE}' does not exist. Please check the BASE path.")


Listing contents of: /Users/annedranowski/HQ/🔬 Research/Untangling Number/crossing-reduction/
  Directories:
    - tb_knots/
    - __pycache__/
    - .cache/
    - runs/
  Files:
    - ppo_knot_rl_spherogram_continued.zip
    - .DS_Store
    - small-jones.txt
    - run_variant_sweep.py
    - random_diagrams.csv
    - ppo_100000_steps.zip
    - generated_pd_backtrack.txt
    - generated_T_pd_backtrack.txt
    - ppo_50000_steps.zip
    - random_diagrams_2.csv
    - 3-16.txt
    - best_model.zip
    - very_hard_unknots_2.csv


In [6]:
# --------- 7) Train PPO ----------

# === Envs ===
vec_env  = DummyVecEnv([lambda: make_env()])
eval_env = DummyVecEnv([lambda: make_env()])

seed = random.randint(1, 100)
SEED = random.seed(seed)
TOTAL_STEPS = 12288*1

# === Load existing model or start fresh ===
if os.path.exists(BEST_MODEL_PATH):
    print(f"[Resume] Loading {BEST_MODEL_PATH}")
    model = PPO.load(BEST_MODEL_PATH, env=vec_env, device="auto")
    model.set_logger(configure(TB_DIR, ["stdout", "tensorboard"]))
else:
    print("[Fresh] Starting new PPO model")
    model = PPO(
        "MlpPolicy",
        env,
        learning_rate=lambda p: 3e-4 * p, # linear decay
        n_steps=2048,
        batch_size=256,
        n_epochs=10,
        gamma=0.995,
        gae_lambda=0.97,
        clip_range=0.2,
        ent_coef=0.01,
        vf_coef=0.5,
        max_grad_norm=0.5,
        tensorboard_log=TB_DIR,
        seed=SEED,
    )

# === Callbacks ===
# EvalCallback will now save directly as /MyDrive/crossing-reduction/best_model.zip
eval_cb = EvalCallback(
    eval_env,
    best_model_save_path=BASE,
    eval_freq=10000,
    n_eval_episodes=1,
    deterministic=True,
)
# (optional) periodic checkpoints — also in flat folder
ckpt_cb = CheckpointCallback(save_freq=50000, save_path=BASE, name_prefix="ppo")

# === Train (continues if loaded) ===
model.learn(
    total_timesteps=TOTAL_STEPS,
    callback=[eval_cb, ckpt_cb],
    reset_num_timesteps=False,
    progress_bar=False, # Set to False to avoid LiveError in Colab
)

# === Save snapshot ===
final_path = f"{BASE}/ppo_knot_rl_spherogram_continued.zip"
model.save(final_path)
print(f"[Train] Saved -> {final_path}")

[Resume] Loading /Users/annedranowski/HQ/🔬 Research/Untangling Number/crossing-reduction/best_model.zip
Logging to /Users/annedranowski/HQ/🔬 Research/Untangling Number/crossing-reduction/tb_knots
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 363      |
|    ep_rew_mean     | 626      |
| time/              |          |
|    fps             | 513      |
|    iterations      | 1        |
|    time_elapsed    | 3        |
|    total_timesteps | 142048   |
---------------------------------
-------------------------------------------
| rollout/                |               |
|    ep_len_mean          | 363           |
|    ep_rew_mean          | 626           |
| time/                   |               |
|    fps                  | 489           |
|    iterations           | 2             |
|    time_elapsed         | 8             |
|    total_timesteps      | 144096        |
| train/                  |               |
|    approx_kl          

/Users/annedranowski/HQ/🔬 Research/Untangling Number/.venv/lib/python3.13/site-packages/stable_baselines3/common/evaluation.py:70: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


New best mean reward!
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 363      |
|    ep_rew_mean     | 626      |
| time/              |          |
|    fps             | 581      |
|    iterations      | 5        |
|    time_elapsed    | 17       |
|    total_timesteps | 150240   |
---------------------------------
-------------------------------------------
| rollout/                |               |
|    ep_len_mean          | 363           |
|    ep_rew_mean          | 626           |
| time/                   |               |
|    fps                  | 646           |
|    iterations           | 6             |
|    time_elapsed         | 19            |
|    total_timesteps      | 152288        |
| train/                  |               |
|    approx_kl            | 1.3811048e-05 |
|    clip_fraction        | 0             |
|    clip_range           | 0.2           |
|    entropy_loss         | -2.84         |
|    explained_varianc

In [7]:
L = [[15,116,16,1], [14,62,15,61], [38,2,39,1], [37,21,38,20], [42,21,43,22], [67,26,68,27], [72,28,73,27], [18,92,19,91], [68,97,69,98], [71,99,72,98], [103,101,104,100], [85,71,86,70], [84,109,85,110], [82,107,83,108], [80,105,81,106], [41,2,42,3], [90,18,91,17], [114,20,115,19], [4,24,5,23], [22,4,23,3], [24,6,25,5], [48,101,49,102], [95,7,96,6], [99,105,100,104], [115,16,116,17], [39,12,40,13], [10,60,11,59], [66,58,67,57], [58,10,59,9], [112,89,113,90], [69,87,70,86], [34,111,35,112], [45,111,46,110], [60,14,61,13], [11,40,12,41], [64,30,65,29], [102,49,103,50], [25,8,26,9], [113,93,114,92], [96,7,97,8], [108,83,109,84], [106,81,107,82], [46,79,47,80], [33,79,34,78], [75,28,76,29], [73,56,74,57], [54,66,55,65], [55,74,56,75], [50,47,51,48], [51,33,52,32], [93,36,94,37], [94,44,95,43], [30,64,31,63], [31,76,32,77], [88,36,89,35], [87,44,88,45], [53,62,54,63], [52,78,53,77]]

In [8]:
# -------- Evaluate model on the provided L only --------

# Assuming L is already defined from cell 25BDbyGtEkNu as a list of quads
PD_K = L # L is already a list of quads

# --- Required helper functions and definitions for this cell ---

# ANSI color codes for Colab
RED   = "\033[91m"
RESET = "\033[0m"

def pd_list_to_str(pd_list):
    """List of quads -> JSON string '[[a,b,c,d],...]' for SphKnotEnv."""

    return json.dumps(pd_list)

def make_single_env(pd_list, cfg_local):
    """
    Build a DummyVecEnv whose SphKnotEnv always starts from this single PD.
    """
    pd_str = pd_list_to_str(pd_list)
    pd_lines_single = [pd_str]

    def _make():
        return SphKnotEnv(pd_lines_single, cfg_local)

    return DummyVecEnv([_make])

def run_unknotter_on_pd(pd_list, model, cfg_local, episodes: int = 3, label: str = "") -> bool:
    """
    Return True iff the PPO agent manages to unknot this PD
    (crossings == 0) in at least one of `episodes` runs.

    Prints, after each episode, whether the unknot was found or not.
    SUCCESS lines are printed in red.
    """
    vec_env = make_single_env(pd_list, cfg_local)
    success = False

    for ep in range(episodes):
        obs = vec_env.reset()
        last_cr = None
        steps_taken = 0

        for step in range(cfg_local.max_steps):
            action, _ = model.predict(obs, deterministic=True)
            obs, rewards, dones, infos = vec_env.step(action)

            info = infos[0]
            cr = info.get("crossings", None)
            if cr is not None:
                last_cr = cr
            steps_taken = step + 1

            if cr == 0:  # unknot reached
                success = True
                break

            if dones[0]:
                break

        if success:
            print(
                f"{RED}    [{label}] episode {ep+1}/{episodes}: "
                f"SUCCESS (unknotted in {steps_taken} steps){RESET}"
            )
            break
        else:
            print(
                f"    [{label}] episode {ep+1}/{episodes}: no unknot "
                f"(final crossings = {last_cr})"
            )

    vec_env.close()
    return success

# --- End of helper functions ---


print(f"\nEvaluating model on the given L (crossings: {len(PD_K)})\n")

success = run_unknotter_on_pd(
    PD_K,
    model,
    cfg,
    episodes=1, # Run for one episode to check unknotting
    label="L"
)

# ---- color logic: red if unknotting was found ----
# RED and RESET are defined above within this cell's scope

line = (
    f"Provided L (crossings={len(PD_K)}): "
    f"Model can unknot: {success}"
)

if success:
    print(RED + line + RESET)
else:
    print(line)


Evaluating model on the given L (crossings: 58)

    [L] episode 1/1: SUCCESS (unknotted in 8 steps)
Provided L (crossings=58): Model can unknot: True


In [9]:
#K1 = [[1,9,2,8],[3,11,4,10],[5,13,6,12],[7,1,8,14],[9,3,10,2],[11,5,12,4],[13,7,14,6]] This is 7_1
#K2 = [[8,1,9,2],[10,3,11,4],[12,5,13,6],[14,7,1,8],[2,9,3,10],[4,11,5,12],[6,13,7,14]] This is the mirror of 7_1

#K1 = [[2,8,3,7],[4,10,5,9],[6,2,7,1],[8,4,9,3],[10,6,1,5]] # This is 5_1
#K2 = [[1,10,2,11],[3,13,4,12],[5,15,6,14],[7,1,8,16],[9,2,10,3],[11,9,12,8],[13,5,14,4],[15,7,16,6]] # This is 8_2

K1 = [[4,2,5,1],[8,6,1,5],[6,3,7,4],[2,7,3,8]] # This is 4_1
K2 = [[1,13,2,12],[3,11,4,10],[5,17,6,16],[7,15,8,14],[9,1,10,18],[11,3,12,2],[13,5,14,4],[15,7,16,6],[17,9,18,8]] # This is 9_10

# ---------- connected sum (your build uses .connected_sum) ----------
def connected_sum_pd(PD1, PD2, simplify=True):
    L1, L2 = Link(PD1), Link(PD2)
    out = L1.connected_sum(L2)  # returns new Link or mutates in place
    L = out if out is not None else L1
    if simplify and hasattr(L, "simplify"):
        try:
            L.simplify()
        except TypeError:
            pass
    return [list(q) for q in L.PD_code()]  # normalize to list-of-lists

T = connected_sum_pd(K1,K2)
T

[[11, 17, 12, 16],
 [15, 13, 16, 12],
 [13, 18, 14, 19],
 [17, 14, 18, 15],
 [19, 5, 20, 4],
 [21, 3, 22, 2],
 [23, 9, 24, 8],
 [25, 7, 0, 6],
 [1, 11, 2, 10],
 [3, 21, 4, 20],
 [5, 23, 6, 22],
 [7, 25, 8, 24],
 [9, 1, 10, 0]]

In [10]:
# -----------------------------------------
# Randomize knot T via backtrack + RIII (larger deduplicated pool)
# -----------------------------------------

# Generation targets
NUM_VARIANTS_FOR_T = 16          # number of unique randomised diagrams to keep
MAX_ATTEMPTS_FOR_T = 5000         # safety cap so this cell always terminates

# Backtrack/randomisation controls
BACKTRACK_STEPS_MIN = 1
BACKTRACK_STEPS_MAX = 4
RIII_STEPS_MAX      = 32

# Keep only diagrams in a crossing window near T to stay 'relevant'
MIN_CROSSINGS_KEEP = 15
MAX_CROSSINGS_KEEP = 22

GEN_T_OUT_PATH = f"{BASE}/generated_T_pd_backtrack.txt"

print("Randomising knot T:")

generated_T_pd_lines = []
seen_pd_strings = set()

L0_T = Link(T)
base_crossings = len(L0_T.crossings)
print(f"  base crossings(T) = {base_crossings}")

accepted = 0
attempts = 0

while accepted < NUM_VARIANTS_FOR_T and attempts < MAX_ATTEMPTS_FOR_T:
    attempts += 1

    # fresh copy from PD code so we don't mutate the base T
    pd0 = L0_T.PD_code()
    if isinstance(pd0, list):
        L = Link(pd0)
    else:
        L = Link([[int(getattr(e, "label", e)) for e in vtx] for vtx in pd0])

    steps = random.randint(BACKTRACK_STEPS_MIN, BACKTRACK_STEPS_MAX)
    L.backtrack(steps=steps, prob_type_1=0.35, prob_type_2=0.65)
    L, _ = riii_shuffle_only_link(L, min(steps, RIII_STEPS_MAX))

    pd_obj = L.PD_code()
    if isinstance(pd_obj, list):
        quads = [[int(x) for x in quad] for quad in pd_obj]
    else:
        quads = [[int(getattr(edge, "label", edge)) for edge in vtx] for vtx in pd_obj]

    c = len(quads)
    if c < MIN_CROSSINGS_KEEP or c > MAX_CROSSINGS_KEEP:
        continue

    pd_str_new = json.dumps(quads, separators=(",",":"))
    if pd_str_new in seen_pd_strings:
        continue

    seen_pd_strings.add(pd_str_new)
    generated_T_pd_lines.append(pd_str_new)
    accepted += 1

    if accepted % 25 == 0 or accepted == 1:
        print(f"  accepted {accepted:4d}/{NUM_VARIANTS_FOR_T} after {attempts:5d} attempts (last: steps={steps}, crossings={c})")

print(f"\nGenerated {len(generated_T_pd_lines)} unique randomised PDs from T (attempts={attempts}).")
if len(generated_T_pd_lines) < NUM_VARIANTS_FOR_T:
    print("[WARN] Hit MAX_ATTEMPTS_FOR_T before reaching NUM_VARIANTS_FOR_T.")

with open(GEN_T_OUT_PATH, "w") as f:
    for line in generated_T_pd_lines:
        f.write(line + "\n")

print(f"Wrote {len(generated_T_pd_lines)} PD lines for T to {GEN_T_OUT_PATH}")

peek_n = min(4, len(generated_T_pd_lines))
print(f"\nPeeking at the first {peek_n} generated PDs for T:\n")
for i, pd_str in enumerate(generated_T_pd_lines[:peek_n]):
    quads = json.loads(pd_str)
    try:
        c = len(Link(quads).crossings)
    except Exception:
        c = "?"
    print(f"[{i}] crossings = {c}")
    print(f"    {pd_str}")


Randomising knot T:
  base crossings(T) = 13
  accepted    1/16 after     2 attempts (last: steps=4, crossings=21)

Generated 16 unique randomised PDs from T (attempts=19).
Wrote 16 PD lines for T to /Users/annedranowski/HQ/🔬 Research/Untangling Number/crossing-reduction/generated_T_pd_backtrack.txt

Peeking at the first 4 generated PDs for T:

[0] crossings = 21
    [[27,21,28,20],[21,27,22,26],[25,18,26,19],[19,22,20,23],[17,37,18,36],[15,39,16,38],[9,31,10,30],[3,33,4,32],[39,29,40,28],[37,17,38,16],[35,15,36,14],[31,7,32,6],[29,1,30,0],[4,12,5,11],[5,10,6,11],[41,24,0,25],[40,24,41,23],[8,2,9,1],[7,2,8,3],[34,13,35,14],[33,13,34,12]]
[1] crossings = 15
    [[17,13,18,12],[13,17,14,16],[15,10,16,11],[11,14,12,15],[9,25,10,24],[5,29,6,28],[3,21,4,20],[1,23,2,22],[29,19,0,18],[25,9,26,8],[23,5,24,4],[21,3,22,2],[19,1,20,0],[7,27,8,26],[6,27,7,28]]
[2] crossings = 19
    [[21,17,22,16],[17,21,18,20],[19,14,20,15],[15,18,16,19],[13,35,14,34],[11,37,12,36],[5,27,6,26],[1,31,2,30],[37,23,

In [11]:
PD_PATH2 = GEN_T_OUT_PATH

In [12]:
# ================================# 1) Read first PDs from PD_PATH2# ================================
def read_pd_lines_from_file(path: str, max_lines: int | None = None):
    """Read non-empty lines from a local PD file and return the first `max_lines`."""
    with open(path, "r") as f:
        lines = [ln.strip() for ln in f if ln.strip()]
    return lines[:max_lines]

def parse_pd_line_to_list(line: str):
    """
    Parse either
      - JSON PD lists: '[[1,5,2,4],[3,1,4,6],...]'
      - or PD[X[1,5,2,4], X[3,1,4,6], ...]
    into [[a,b,c,d], ...].
    """
    t = line.strip()

    # 1) Try JSON / Python list first
    try:
        obj = json.loads(t)
        if isinstance(obj, list) and obj and isinstance(obj[0], (list, tuple)):
            quads = []
            for quad in obj:
                if len(quad) != 4:
                    raise ValueError(f"PD block must have 4 integers, got {quad!r}")
                quads.append([int(x) for x in quad])
            return quads
    except Exception:
        pass  # not JSON, fall back to PD[X[...]] format

    # 2) Fallback: PD[X[...], X[...], ...] format
    items = re.findall(r'[Xx]\s*\[([^\]]+)\]', t)
    if not items:
        raise ValueError(f"No X[...] blocks found in line: {line!r}")

    quads = []
    for it in items:
        nums = [int(x.strip()) for x in it.split(',')]
        if len(nums) != 4:
            raise ValueError(f"PD block must have 4 integers, got {nums!r}")
        quads.append(nums)
    return quads

# Assuming PD_PATH2 is defined elsewhere, e.g., in an earlier cell
# PD_PATH2 = f"{BASE}/generated_T_pd_backtrack.txt"
raw_pd_lines = read_pd_lines_from_file(PD_PATH2, max_lines=None)
print(f"Loaded {len(raw_pd_lines)} PD lines from PD_PATH2")
preview_n = min(5, len(raw_pd_lines))
print(f"First {preview_n} PD lines from PD_PATH2:")
for i, s in enumerate(raw_pd_lines[:preview_n]):
    print(f"  [{i}] {s}")

pd_lists = [parse_pd_line_to_list(s) for s in raw_pd_lines]
print("\nParsed as PD lists (lengths):", [len(p) for p in pd_lists])

# ================================# 2) Crossing flip + PD variants# ================================
def flip_crossing_quad(quad):
    """[a,b,c,d] -> [b,c,d,a]"""
    a, b, c, d = quad
    return [b, c, d, a]

def pd_list_to_str(pd_list):
    """
    Stringify to something parse_link_strict accepts.
    This produces '[[1,5,2,4],[3,1,4,6],...]' which matches the
    _RE_PDLIST branch in parse_link_strict.
    """
    return json.dumps(pd_list)

def generate_changed_pds(pd_list, k):
    """
    Yield all PD-lists obtained by flipping exactly k crossings,
    in lexicographic order on crossing indices.
    """
    n = len(pd_list)
    for idxs in itertools.combinations(range(n), k):
        idxs_set = set(idxs)
        new_pd = []
        for i, quad in enumerate(pd_list):
            if i in idxs_set:
                new_pd.append(flip_crossing_quad(quad))
            else:
                new_pd.append(list(quad))
        yield idxs, new_pd

# ================================# 3) RL wrapper: run agent on one PD# ================================
# Load your best model (avoid reloading if already in memory)
try:
    model
except NameError:
    model = PPO.load(BEST_MODEL_PATH, device="cpu")
    print("\n[Model] Loaded PPO model from:", BEST_MODEL_PATH)

def make_single_env(pd_list, cfg: EnvCfg):
    """
    Build a DummyVecEnv whose SphKnotEnv always starts from this single PD.
    """
    pd_str = pd_list_to_str(pd_list)
    pd_lines_single = [pd_str]

    def _make():
        return SphKnotEnv(pd_lines_single, cfg)

    return DummyVecEnv([_make])

def run_unknotter_on_pd(pd_list, model, cfg: EnvCfg,
                        episodes: int = 3, label: str = "") -> tuple[bool, int]:
    """
    Return (True, 0) iff the PPO agent manages to unknot this PD
    in at least one of `episodes` runs. Otherwise, returns
    (False, minimum_crossings_seen_across_rollout).
    """
    vec_env = make_single_env(pd_list, cfg)
    success = False
    min_crossings = len(pd_list)

    for ep in range(episodes):
        obs = vec_env.reset()  # no info at vec level
        last_cr = min_crossings
        for step in range(cfg.max_steps):
            action, _ = model.predict(obs, deterministic=True)
            obs, rewards, dones, infos = vec_env.step(action)

            info = infos[0]
            cr = info.get("crossings", None)
            if cr is not None:
                last_cr = cr
                min_crossings = min(min_crossings, cr)

            # We declare success if the environment reports crossings == 0
            if cr == 0:
                success = True
                min_crossings = 0
                break

            if dones[0]:
                break

        if success:
            break
        else:
            if last_cr is not None:
                min_crossings = min(min_crossings, last_cr)

    vec_env.close()
    return success, min_crossings

# ================================# 4) Untangling number search# ================================
def compute_untangling_number(pd_list,
                              model,
                              cfg: EnvCfg,
                              max_changes: int | None = None,
                              episodes: int = 3):
    """
    Search for minimal k \u2265 1 such that flipping k crossings
    yields a diagram which the RL machine can unknot.

    Returns (k, idxs) where
      - k is the untangling number (int or None)
      - idxs is the tuple of crossing indices to flip (or None)
    """
    n = len(pd_list)
    if max_changes is None:
        max_changes = n

    for k in range(1, max_changes + 1):
        for idxs, new_pd in generate_changed_pds(pd_list, k):
            success, _ = run_unknotter_on_pd(new_pd, model, cfg, episodes=episodes)
            if success:
                return k, idxs

    return None, None


Loaded 16 PD lines from PD_PATH2
First 5 PD lines from PD_PATH2:
  [0] [[27,21,28,20],[21,27,22,26],[25,18,26,19],[19,22,20,23],[17,37,18,36],[15,39,16,38],[9,31,10,30],[3,33,4,32],[39,29,40,28],[37,17,38,16],[35,15,36,14],[31,7,32,6],[29,1,30,0],[4,12,5,11],[5,10,6,11],[41,24,0,25],[40,24,41,23],[8,2,9,1],[7,2,8,3],[34,13,35,14],[33,13,34,12]]
  [1] [[17,13,18,12],[13,17,14,16],[15,10,16,11],[11,14,12,15],[9,25,10,24],[5,29,6,28],[3,21,4,20],[1,23,2,22],[29,19,0,18],[25,9,26,8],[23,5,24,4],[21,3,22,2],[19,1,20,0],[7,27,8,26],[6,27,7,28]]
  [2] [[21,17,22,16],[17,21,18,20],[19,14,20,15],[15,18,16,19],[13,35,14,34],[11,37,12,36],[5,27,6,26],[1,31,2,30],[37,23,0,22],[35,13,36,12],[31,11,32,10],[29,3,30,2],[23,1,24,0],[24,3,25,4],[25,5,26,4],[6,27,7,28],[7,29,8,28],[9,33,10,32],[8,33,9,34]]
  [3] [[27,17,28,16],[17,25,18,24],[23,12,24,13],[15,18,16,19],[11,37,12,36],[9,39,10,38],[5,33,6,32],[1,35,2,34],[39,29,0,28],[37,11,38,10],[35,9,36,8],[33,5,34,4],[29,1,30,0],[21,30,22,31],[22,32,23,

# Task
Iterate through each knot in the `PD_PATH2` file. For each knot, randomly select ~~5~~ 3 distinct crossings and apply the `flip_crossing_quad` function. Then, test if the RL model can unknot each of these modified knots using the `run_unknotter_on_pd` function, creating a single-knot environment for each test. Finally, summarize the results for each original knot by displaying its initial crossing count, the 5 specific crossings that were flipped, and whether the RL model successfully unknotted the resulting diagram.

## Iterate Knots and Apply Flips

### Subtask:
Loop through each PD list previously loaded from `PD_PATH2`. For each knot, randomly select ~~5~~ 3 distinct crossings and apply the `flip_crossing_quad` function. The indices of the flipped crossings will be reported.


**Reasoning**:
The subtask requires iterating through each PD list, randomly selecting ~~5~~ 3 crossings, flipping them, and storing the original, flipped, and index information. This code block will perform these operations as specified.



In [13]:
original_pd_list = f"{BASE}/generated_T_pd_backtrack.txt"  # file with generated JSON PD lists

In [14]:
flips = 3  # Number of crossings to flip, as per user's 20 choose 3 example

all_possible_modified_knots_info = []

# Assuming pd_lists contains the original PD lists to work with
# In this specific context, pd_lists has only one element (the randomized T)
# So, we'll iterate through each original_pd_list from pd_lists
for original_pd_list in pd_lists:
    n_crossings = len(original_pd_list)

    # Ensure there are enough crossings to perform the requested number of flips
    if n_crossings < flips:
        print(f"Skipping PD list with only {n_crossings} crossings (less than {flips} required flips).")
        continue

    # Generate all unique combinations of 'flips' distinct crossing indices
    for flipped_indices_combo in itertools.combinations(range(n_crossings), flips):
        flipped_indices = sorted(list(flipped_indices_combo)) # Ensure sorted for consistent reporting

        flipped_pd_list = []
        for i, quad in enumerate(original_pd_list):
            if i in flipped_indices:
                flipped_pd_list.append(flip_crossing_quad(quad))
            else:
                flipped_pd_list.append(list(quad))  # Ensure a deep copy

        all_possible_modified_knots_info.append({
            "original_pd_list": original_pd_list,
            "flipped_pd_list": flipped_pd_list,
            "flipped_indices": tuple(flipped_indices)
        })

print(f"Generated {len(all_possible_modified_knots_info)} all possible modified knot variants (combinations of {flips} flips).")
# for i, knot_info in enumerate(all_possible_modified_knots_info):
#     print(f"Knot {i}: original crossings = {len(knot_info['original_pd_list'])}, flipped indices = {knot_info['flipped_indices']}")

print("\nEvaluating RL model on all possible modified knots:")

results_summary_all_possible = []

for i, knot_info in enumerate(all_possible_modified_knots_info):
    original_pd_list = knot_info["original_pd_list"]
    flipped_pd_list = knot_info["flipped_pd_list"]
    flipped_indices = knot_info["flipped_indices"]

    initial_crossings = len(original_pd_list)

    print(f"\n--- Knot variant {i+1}/{len(all_possible_modified_knots_info)} ---")
    print(f"  Original crossings: {initial_crossings}")
    print(f"  Flipped indices: {flipped_indices}")

    # Run the unknotter on the flipped PD list
    success, min_crossings_found = run_unknotter_on_pd(
        flipped_pd_list,
        model,
        cfg,
        episodes=1, # Use 1 episode for evaluation speed
        label=f"Knot {i+1} (flips {flipped_indices})"
    )

    print(f"  Min crossings found: {min_crossings_found}")

    results_summary_all_possible.append({
        "knot_index": i + 1,
        "original_crossings": initial_crossings,
        "flipped_indices": flipped_indices,
        "rl_unknot_success": success,
        "min_crossings_found": min_crossings_found
    })

print("\n=================== Final Summary (All Possible Flips) ===================")
for res in results_summary_all_possible:
    status_color = RED if res["rl_unknot_success"] else RESET
    print(f"{status_color}Knot {res['knot_index']}: Original Crossings={res['original_crossings']}, Flipped Indices={res['flipped_indices']}, RL Unknot Success={res['rl_unknot_success']}, Min Crossings Found={res['min_crossings_found']}{RESET}")
print("========================================================================")

Generated 13497 all possible modified knot variants (combinations of 3 flips).

Evaluating RL model on all possible modified knots:

--- Knot variant 1/13497 ---
  Original crossings: 21
  Flipped indices: (0, 1, 2)
  Min crossings found: 9

--- Knot variant 2/13497 ---
  Original crossings: 21
  Flipped indices: (0, 1, 3)
  Min crossings found: 9

--- Knot variant 3/13497 ---
  Original crossings: 21
  Flipped indices: (0, 1, 4)
  Min crossings found: 10

--- Knot variant 4/13497 ---
  Original crossings: 21
  Flipped indices: (0, 1, 5)
  Min crossings found: 10

--- Knot variant 5/13497 ---
  Original crossings: 21
  Flipped indices: (0, 1, 6)
  Min crossings found: 10

--- Knot variant 6/13497 ---
  Original crossings: 21
  Flipped indices: (0, 1, 7)
  Min crossings found: 10

--- Knot variant 7/13497 ---
  Original crossings: 21
  Flipped indices: (0, 1, 8)
  Min crossings found: 10

--- Knot variant 8/13497 ---
  Original crossings: 21
  Flipped indices: (0, 1, 9)
  Min crossings 

KeyboardInterrupt: 

In [ ]:
filtered_results = [res for res in results_summary_all_possible if res["min_crossings_found"] <= 3]
print(f"\n=================== {len(filtered_results)}/{len(results_summary_all_possible)} with Min Crossings <= 3) ===================")

if filtered_results:
    for res in filtered_results:
        status_color = RED if res["rl_unknot_success"] else RESET
        print(f"{status_color}Knot {res['knot_index']}: Original Crossings={res['original_crossings']}, Flipped Indices={res['flipped_indices']}, RL Unknot Success={res['rl_unknot_success']}, Min Crossings Found={res['min_crossings_found']}{RESET}")
else:
    print("No knots found with min crossings <= 3.")
print("==========================================================================")


In [ ]:
import matplotlib.pyplot as plt

# Extract all min_crossings_found values
min_crossings_values = [res['min_crossings_found'] for res in results_summary_all_possible]

plt.figure(figsize=(12, 6))
plt.hist(min_crossings_values, bins=range(0, max(min_crossings_values) + 2), edgecolor='black')
plt.title('Distribution of Minimum Crossings Found by RL Model')
plt.xlabel('Minimum Crossings Found')
plt.ylabel('Frequency')
plt.xticks(range(0, max(min_crossings_values) + 1))
plt.grid(axis='y', alpha=0.75)
plt.show()

# Optionally, print some basic statistics
print(f"\nTotal data points: {len(min_crossings_values)}")
print(f"Mean min crossings: {np.mean(min_crossings_values):.2f}")
print(f"Median min crossings: {np.median(min_crossings_values)}")
print(f"Max min crossings: {np.max(min_crossings_values)}")
print(f"Min min crossings (excluding 0 if any): {np.min([x for x in min_crossings_values if x > 0]) if any(x > 0 for x in min_crossings_values) else 'N/A'}")
print(f"Number of unknotted knots (0 crossings): {min_crossings_values.count(0)}")


**No need to run the below for now!**

In [ ]:
flips = 3
num_flips_per_knot = 1 # Define k, the number of times to apply flips random flips per knot
modified_knots_info = []

for original_pd_list in pd_lists:
    n_crossings = len(original_pd_list)

    # Ensure there are at least flips crossings to flip
    if n_crossings < flips:
        print(f"Skipping PD list with only {n_crossings} crossings (less than flips required flips).")
        continue

    for _ in range(num_flips_per_knot):
        # Randomly select flips distinct crossing indices to flip
        flipped_indices = random.sample(range(n_crossings), flips)
        flipped_indices.sort() # Sort for consistent reporting

        flipped_pd_list = []
        for i, quad in enumerate(original_pd_list):
            if i in flipped_indices:
                flipped_pd_list.append(flip_crossing_quad(quad))
            else:
                flipped_pd_list.append(list(quad)) # Ensure a deep copy

        modified_knots_info.append({
            "original_pd_list": original_pd_list,
            "flipped_pd_list": flipped_pd_list,
            "flipped_indices": tuple(flipped_indices)
        })

print(f"Generated {len(modified_knots_info)} modified knot variants.")
for i, knot_info in enumerate(modified_knots_info):
    print(f"Knot {i}: original crossings = {len(knot_info['original_pd_list'])}, flipped indices = {knot_info['flipped_indices']}")

In [ ]:
print("\nEvaluating RL model on modified knots:")

results_summary = []

for i, knot_info in enumerate(modified_knots_info):
    original_pd_list = knot_info["original_pd_list"]
    flipped_pd_list = knot_info["flipped_pd_list"]
    flipped_indices = knot_info["flipped_indices"]

    initial_crossings = len(original_pd_list)

    print(f"\n--- Knot variant {i+1}/{len(modified_knots_info)} ---")
    print(f"  Original crossings: {initial_crossings}")
    print(f"  Flipped indices: {flipped_indices}")

    # Run the unknotter on the flipped PD list
    success, min_crossings_found = run_unknotter_on_pd(
        flipped_pd_list,
        model,
        cfg,
        episodes=1, # Use 1 episode for evaluation speed
        label=f"Knot {i+1} (flips {flipped_indices})"
    )

    print(f"  Min crossings found: {min_crossings_found}")

    results_summary.append({
        "knot_index": i + 1,
        "original_crossings": initial_crossings,
        "flipped_indices": flipped_indices,
        "rl_unknot_success": success,
        "min_crossings_found": min_crossings_found
    })

print("\n=================== Final Summary ===================")
for res in results_summary:
    status_color = RED if res["rl_unknot_success"] else RESET
    print(f"{status_color}Knot {res['knot_index']}: Original Crossings={res['original_crossings']}, Flipped Indices={res['flipped_indices']}, RL Unknot Success={res['rl_unknot_success']}, Min Crossings Found={res['min_crossings_found']}{RESET}")
print("=====================================================")